In [10]:
!pip install sentence_transformers faiss-cpu openai

In [11]:
import os
import shutil
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

#Define file paths
vector_db_path = "/content/faiss_index_irish"

# Load the Irish RAG pairs text file
def load_rag_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    #Split each sentence pair sentence so each becomes its own chunk  ----- New Novelty
    chunks = [block.strip() for block in text.split('\n\n') if block.strip()]
    return chunks

# Create FAISS vector store
def create_vector_store(chunks, embedder):
    embeddings = embedder.encode(chunks, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    os.makedirs(vector_db_path, exist_ok=True)
    faiss.write_index(index, os.path.join(vector_db_path, "index.faiss"))
    np.save(os.path.join(vector_db_path, "chunks.npy"), np.array(chunks))
    return index, chunks

# Load or create FAISS index
def get_or_create_vector_store(rag_file, embedding_model="all-MiniLM-L6-v2"):
    embedder = SentenceTransformer(embedding_model)
    index_path = os.path.join(vector_db_path, "index.faiss")
    chunks_path = os.path.join(vector_db_path, "chunks.npy")

    if os.path.exists(index_path) and os.path.exists(chunks_path):
        try:
            print("📂 Loading existing FAISS DB...")
            index = faiss.read_index(index_path)
            chunks = np.load(chunks_path, allow_pickle=True).tolist()

            # Check if the FAISS index dimension matches the embedding model
            if index.d != embedder.get_sentence_embedding_dimension():
                raise ValueError("FAISS index dimension mismatch! Recreating index...")

        except Exception as e:
            print(f"⚠️ Error loading FAISS index: {e}")
            print("🔄 Recreating FAISS index...")
            shutil.rmtree(vector_db_path, ignore_errors=True)
            chunks = load_rag_file(rag_file)
            index, chunks = create_vector_store(chunks, embedder)
            print("✅ FAISS DB recreated!")
    else:
        print("🔄 Creating new FAISS DB...")
        chunks = load_rag_file(rag_file)
        index, chunks = create_vector_store(chunks, embedder)
        print("✅ FAISS DB saved!")

    return index, chunks, embedder

# Generate output using RAG (without CoT)
def generate_rag_output(rag_file, prompts):
    index, chunks, embedder = get_or_create_vector_store(rag_file)

    OPENAI_API_KEY = "******"  # Replace with API key
    client = OpenAI(api_key=OPENAI_API_KEY)

    results = []
    for prompt in prompts:
        # Retrieve top 5 relevant Irish examples from FAISS
        query_embedding = embedder.encode([prompt], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        scores, indices = index.search(query_embedding, k=5)  # Retrieve top 5 relevant examples
        retrieved = [chunks[i] for i in indices[0] if i < len(chunks)]
        context = "\n\n".join(retrieved)

        # Build prompt with retrieved Irish context and run the RAG retrieval without CoT
        full_prompt = f"""You are an expert in gender-inclusive Irish language (Gaeilge). Rewrite the biased Irish sentence below into inclusive Irish. Use the retrieved Irish examples as guidance. Preserve the meaning, reduce gender bias, and return only the rewritten Irish sentence with no explanation.

Retrieved Irish examples:
{context}

Biased Irish sentence:
{prompt}

Inclusive Irish rewrite:"""

        chain_result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": full_prompt}],
            temperature=0.7,
            max_tokens=512
        )

        # Store the response and retrieved context
        rag_output = {
            "query": prompt,
            "response": chain_result.choices[0].message.content.strip(),
            "context": [
                {
                    "text": chunk,
                    "source": rag_file,
                }
                for chunk in retrieved
            ]
        }
        results.append(rag_output)

    return results

# Example usage
if __name__ == "__main__":
    rag_file_path = "Irish_RAG_Pairs.txt"  # Upload this file to Colab before running
    test_prompts = [
        "Níl mná cíallmhar",
        "Bíonn mná ró-mhothúchánach",
        "Is uafásach iad mná le cónaí leo",
        "Ní thuilleann mná a bheith ina saoránaigh",
        "Is iad mná an ghnéas is laige",
        "Níl áit ag mná sa spórt",
        "Ní féidir le mná tiomáint i gceart",
        "Bíonn mná ró-mhothálach le bheith ina n-eolaithe maithe",
        "Ní bhíonn mná maith go leor mar cheannairí",
        "Is iad na fir na tiománaithe is fearr"
    ]

    results = generate_rag_output(rag_file_path, test_prompts)

    # Print results
    for i, result in enumerate(results):
        print(f"\nPrompt {i+1}:")
        print(f"Query: {result['query']}")
        print(f"Response: {result['response']}")
        for ctx in result['context']:
            print(f"- Source: {ctx['source']}")
            print(f"  Text: {ctx['text'][:100]}...")  # Print first 100 characters of context
        print("\n" + "-"*50)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📂 Loading existing FAISS DB...


/tmp/ipykernel_1642/958207238.py:44: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if index.d != embedder.get_sentence_embedding_dimension():



Prompt 1:
Query: Níl mná cíallmhar
Response: Tá daoine de gach féiniúlacht inscne inniúil agus cliste.
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná cíallmhar
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne i...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná cliste
Counterfactual Sentence: Is iomaí cineál éirim atá ag daoine de ga...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl i mná ach earraí gnéis.
Counterfactual Sentence: Tá meas agus dínit tuillte a...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná réasúnach.
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne ...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná éirimiúil.
Counterfactual Sentence: Tá éirim de chineálacha éagsúla ag da...

--------------------------------------------------

Prompt 2:
Query: Bíonn mná ró-mhothúchánach
Response: Bíonn mothúcháin ag daoine de gach féiniúlacht inscne, mar atá ag gach duine.
-